In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error
import time
#import pygwalker as pyg
from datetime import datetime
import os
#from ydata_profiling import ProfileReport
import csv

In [2]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_18932\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


## Imputing missing values ##

In [3]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [4]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [5]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


In [6]:
df.drop(['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

## Feature Engineering ##

In [7]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [8]:
# walker = pyg.walk(df)

In [9]:
df.sample(5)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionDistance,Promo2,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth
392755,281,4,2014-07-17,6179,539,1,1,0,1,d,...,6970.0,0,2014,7,17,0,34.0,29,0.0,0
274363,700,5,2014-11-21,3994,537,1,0,0,0,a,...,830.0,1,2014,11,21,1,22.0,47,17.0,0
512121,7,6,2014-03-29,5693,598,1,0,0,0,a,...,24000.0,0,2014,3,29,0,11.0,13,0.0,0
310912,801,1,2014-10-13,4085,433,1,0,0,1,d,...,48330.0,0,2014,10,13,0,18.0,42,0.0,0
273536,831,6,2014-11-22,9559,1247,1,0,0,0,a,...,800.0,0,2014,11,22,0,89.0,47,0.0,0


In [10]:
num_col=['Customers','CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths','DayOfWeek','Month','Day']
cat_col = ['StoreType','Assortment','Year']

In [11]:
df = df.sort_values('Date')

cutoff_date = '2015-06-01'

train = df[df['Date'] < cutoff_date]
valid = df[df['Date'] >= cutoff_date]

X_train = train.drop(['Sales', 'Date'], axis=1)
y_train = train['Sales']

X_test = valid.drop(['Sales', 'Date'], axis=1)
y_test = valid['Sales']

In [12]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first'),cat_col)    
])

In [13]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [14]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    "XGBRegressor" : XGBRegressor(tree_method='hist',n_jobs=-1),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=50,max_depth=15,n_jobs=-1),
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    prediction = model.predict(X_test)
    prediction_stop = time.perf_counter()
    prediction_time_taken = prediction_stop-prediction_start
    rmse = root_mean_squared_error(y_test,prediction)
    rmsep = rmse/y_test_mean

    print(f'{model_name}: {rmsep*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        'rmse': rmse,
        'rmsep_percent': rmsep * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmse', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)

XGBRegressor: 12.88%

Training time taken for XGBRegressor: 6.2947

prediction time taken for XGBRegressor: 0.0300

RandomForestRegressor: 14.93%

Training time taken for RandomForestRegressor: 67.2670

prediction time taken for RandomForestRegressor: 0.1188

LinearRegression: 23.37%

Training time taken for LinearRegression: 0.6267

prediction time taken for LinearRegression: 0.0061

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008355 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1087
[LightGBM] [Info] Number of data points in the train set: 949194, number of used features: 14
[LightGBM] [Info] Start training from score 5745.395182
LGBMRegressor: 15.33%

Training time taken for LGBMRegressor: 3.0207

prediction time taken for LGBMRegressor: 0.0637

